# ML Server — Colab Runner

Тонкий ноутбук який запускає ML сервер з git репозиторію.
Весь код пишеться локально у VSCode і пушиться в git.

## Як використовувати:
1. Runtime → Change runtime type → **T4 GPU**
2. Заміни URL у Cell 1 на свій git репо
3. Run all
4. Скопіюй ngrok URL з виводу Cell 4 у `.env` свого backend

Якщо щось зламалось — змінюй код локально, push, потім тут просто `!git pull` і перезапускай Cell 4.

In [ ]:
# Cell 1: Clone or update repo
import os

REPO_URL = "https://github.com/USERNAME/diploma-ml-server.git"  # ← заміни
REPO_DIR = "/content/diploma-ml-server"

if os.path.exists(REPO_DIR):
    print(f"Updating existing repo at {REPO_DIR}")
    !cd {REPO_DIR} && git pull
else:
    print(f"Cloning {REPO_URL}")
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!ls

In [ ]:
# Cell 2: Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Cell 3: Mount Drive (для збереження моделей і завантаження датасетів)
from google.colab import drive
drive.mount('/content/drive')

# Перевіримо що наш DATASETS_ROOT існує
import os
from ml_server.config import DATASETS_ROOT, MODELS_ROOT, CHUNKS_ROOT
for path in [DATASETS_ROOT, MODELS_ROOT, CHUNKS_ROOT]:
    os.makedirs(path, exist_ok=True)
    print(f"  {path}: {'✓' if os.path.exists(path) else '✗'}")

In [ ]:
# Cell 4: (Optional) Set ngrok auth token
# Get free token at https://dashboard.ngrok.com/get-started/your-authtoken
# Without token — ngrok works but session is shorter
import os
os.environ["NGROK_AUTH_TOKEN"] = ""  # ← встав свій ngrok token (опціонально)

In [ ]:
# Cell 5: 🟢 Start server
# Цей блок блокує — поки сервер не зупиниш, він буде слухати запити.
# Скопіюй ngrok URL з виводу у backend .env.
from ml_server.app import start_server
start_server()

## Швидкий debug (опціонально)

Якщо хочеш потестити функцію напряму без HTTP — використовуй scripts/.

In [ ]:
# Опційно: швидкий тест без Flask
# Розкоментуй якщо треба перевірити дані без запуску сервера

# from ml_server.data_loader import build_article_level_data
# train, val, test, full_data, stats, tmpdir = build_article_level_data(
#     dataset_id=1, require_tweets=True
# )
# print(stats)
# print(train.head())

In [ ]:
# Опційно: enable autoreload — зміни в .py файлах підхопляться без рестарту
# Корисно якщо хочеш міняти код прямо у Colab (не рекомендую, краще локально)
# %load_ext autoreload
# %autoreload 2